# Notebook 12.3 -- Tokenization

This notebook builds set of tokens from a text string as in figure 12.8 of the book.

Work through the cells below, running each cell in turn. In various places you will see the words "TO DO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

In [ ]:
import re
import collections

In [ ]:
text = "a sailor went to sea sea sea " +\
    "to see what he could see see see " +\
    "but all that he could see see see " +\
    "was the bottom of the deep blue sea sea sea"

Tokenize the input sentence To begin with the tokens are the individual letters and the </w> whitespace token. So, we represent each word in terms of these tokens with spaces between the tokens to delineate them.

The tokenized text is stored in a structure that represents each word as tokens together with the count of how often that word occurs.  We'll call this the *vocabulary*.

In [ ]:
def initialize_vocabulary(text):
    vocab = collections.defaultdict(int)
    words = text.strip().split()
    for word in words:
        vocab[' '.join(list(word)) + ' </w>'] += 1
    return vocab

In [ ]:
vocab = initialize_vocabulary(text)
print("Vocabulary: {}".format(vocab))
print("Size of vocabulary: {}".format(len(vocab)))

Find all the tokens in the current vocabulary and their frequencies

In [ ]:
def get_tokens_and_frequencies(vocab):
    tokens = collections.defaultdict(int)
    for word, freq in vocab.items():
        word_tokens = word.split()
        for token in word_tokens:
            tokens[token] += freq
    return tokens

In [ ]:
tokens = get_tokens_and_frequencies(vocab)
print("Tokens: {}".format(tokens))
print("Number of tokens: {}".format(len(tokens)))

Find each pair of adjacent tokens in the vocabulary
and count them.  We will subsequently merge the most frequently occurring pair.

In [ ]:
def get_pairs_and_counts(vocab):
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

In [ ]:
pairs = get_pairs_and_counts(vocab)
print("Pairs: {}".format(pairs))
print("Number of distinct pairs: {}".format(len(pairs)))

most_frequent_pair = max(pairs, key=pairs.get)
print("Most frequent pair: {}".format(most_frequent_pair))

Merge the instances of the best pair in the vocabulary

In [ ]:
def merge_pair_in_vocabulary(pair, vocab_in):
    vocab_out = {}
    bigram = re.escape(" ".join(pair))
    p = re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
    for word in vocab_in:
        word_out = p.sub("".join(pair), word)
        vocab_out[word_out] = vocab_in[word]
    return vocab_out

In [ ]:
vocab = merge_pair_in_vocabulary(most_frequent_pair, vocab)
print("Vocabulary: {}".format(vocab))
print("Size of vocabulary: {}".format(len(vocab)))

Update the tokens, which now include the best token 'se'

In [ ]:
tokens = get_tokens_and_frequencies(vocab)
print("Tokens: {}".format(tokens))
print("Number of tokens: {}".format(len(tokens)))

Now let's write the full tokenization routine

In [ ]:
# TODO -- write this routine by filling in this missing parts,
# calling the above routines
def tokenize(text, num_merges):
    # Initialize the vocabulary from the input text
    # vocab = (your code here)

    for i in range(num_merges):
        # Find the tokens and how often they occur in the vocabulary
        # tokens = (your code here)

        # Find the pairs of adjacent tokens and their counts
        # pairs = (your code here)

        # Find the most frequent pair
        # most_frequent_pair = (your code here)
        print('Most frequent pair: {}'.format(most_frequent_pair))

        # Merge the code in the vocabulary
        # vocab = (your code here)

    # Find the tokens and how often they occur in the vocabulary one last time
    # tokens = (your code here)

    return tokens, vocab

In [ ]:
tokens, vocab = tokenize(text, num_merges=22)

In [ ]:
print("Tokens: {}".format(tokens))
print("Number of tokens: {}".format(len(tokens)))
print("Vocabulary: {}".format(vocab))
print("Size of vocabulary: {}".format(len(vocab)))

TODO - Consider the input text:

"How much wood could a woodchuck chuck if a woodchuck could chuck wood"

How many tokens will there be initially and what will they be?
How many tokens will there be if we run the tokenization routine for the maximum number of iterations (merges)?

When you've made your predictions, run the code and see if you are correct.

# Real tokenizers

The routine above is a toy version of byte pair encoding (BPE). Production language models use the same idea, but the merges are learned from a huge corpus, they operate on bytes rather than characters (so any string can be encoded), and the vocabulary has tens or hundreds of thousands of tokens. Here we use the `tiktoken` library, which contains the tokenizers of the GPT family:

* `gpt2`: the tokenizer of GPT-2 (2019).
* `cl100k_base`: the tokenizer of GPT-3.5 and GPT-4.
* `o200k_base`: the tokenizer of GPT-4o.

We encode the same paragraph with each one and look at how it splits a word, a long number, and a few lines of indented Python code.

In [ ]:
# Run this if you're in a Colab to install the tiktoken package
!pip install tiktoken

In [ ]:
import tiktoken

encoding_names = ["gpt2", "cl100k_base", "o200k_base"]
# The first time each encoding is used, its merge table is downloaded (this needs internet)
encodings = {name: tiktoken.get_encoding(name) for name in encoding_names}

paragraph = "Transformers process text as a sequence of tokens. A tokenizer splits the " +\
    "input into pieces that are frequent in the training data, so common words become " +\
    "a single token while rare words are broken into several sub-word units."

number = "3141592653"

code = """def mean(values):
    total = 0
    for v in values:
        total += v
    return total / len(values)
"""

def show_tokens(enc, string):
    # Return the list of token strings that the encoder produces for a string
    ids = enc.encode(string)
    return [enc.decode_single_token_bytes(i).decode("utf-8", errors="replace") for i in ids]

TODO -- Before running the next cell, make some predictions:

1. Which of the three encodings will use the fewest tokens for the paragraph? Why might a larger vocabulary lead to fewer tokens?
2. How many tokens will the word "strawberry" become in each encoding?
3. How will each encoding split the 10-digit number 3141592653 and the indented code (in particular, the runs of spaces)?

When you've made your predictions, run the code and see if you are correct.

In [ ]:
for name, enc in encodings.items():
    print("=" * 60)
    print("Encoding: {}".format(name))
    print("Vocabulary size: {}".format(enc.n_vocab))
    print("Paragraph: {} characters -> {} tokens".format(len(paragraph), len(enc.encode(paragraph))))
    print("'strawberry' -> {}".format(show_tokens(enc, "strawberry")))
    print("'{}' -> {}".format(number, show_tokens(enc, number)))
    code_tokens = show_tokens(enc, code)
    print("Code: {} characters -> {} tokens".format(len(code), len(code_tokens)))
    print(code_tokens)

TODO -- Look at the results and answer these questions:

* The word "strawberry" is split into pieces, so the model never sees its individual letters directly. Why does this make a question like "how many r's are in strawberry?" harder for a language model than it looks?
* How does the treatment of numbers differ between `gpt2` and the newer encodings? Why might splitting numbers into short, regular groups of digits help with arithmetic?
* The older `gpt2` encoding spends many tokens on runs of spaces in the code. Why does this matter for a model that has a fixed context length?